# MarsLandformNet V5 — SCT Expansion via LwF


In [ ]:
!pip install -q transformers peft accelerate timm pillow scikit-learn matplotlib seaborn


In [ ]:
import os
import json
import copy
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f}GB)")
else:
    print("WARNING: No GPU detected. Use Runtime > Change runtime type > T4 GPU.")
print(f"PyTorch: {torch.__version__}, Device: {device}")


In [ ]:
import os, subprocess
from pathlib import Path

# ── Download V5 data from GitHub Release ──
RELEASE_URL = "https://github.com/jejuchild/MarsLab/releases/download/v5-training-data/v5_colab_data.tar.gz"
ARCHIVE = Path("/content/v5_colab_data.tar.gz")
DATA_ROOT = Path("/content/v5_colab_data")

if not DATA_ROOT.exists():
    print("Downloading V5 training data from GitHub Release ...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), RELEASE_URL], check=True)
    print("Extracting ...")
    subprocess.run(["tar", "-xzf", str(ARCHIVE), "-C", "/content"], check=True)
    ARCHIVE.unlink()
    print("Done!")
else:
    print("Data already exists, skipping download.")

# ── Save outputs to Google Drive (optional) ──
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = Path("/content/drive/MyDrive/MarsLab/v5_output")
    print("Checkpoints will be saved to Google Drive.")
except Exception:
    SAVE_ROOT = Path("/content/v5_output")
    print("Drive not mounted. Saving locally (will be lost on disconnect).")

CFG = {
    "model_name": "facebook/dinov2-base",
    "hidden_dim": 768,
    "mola_dim": 25,
    "film_hidden": 64,
    "head_hidden": 128,
    "dropout": 0.4,
    "num_classes": 5,
    "class_names": ["LDA", "LVF", "CCF", "OTHER", "SCT"],
    "class_to_idx": {"LDA": 0, "LVF": 1, "CCF": 2, "OTHER": 3, "SCT": 4},
    "old_class_indices": [0, 1, 2, 3],
    "old_landform_indices": [0, 1, 2],
    "new_class_idx": 4,
    "tile_size": 224,
    "batch_size": 32,
    "num_workers": 2,
    "num_epochs": 60,
    "warmup_epochs": 5,
    "weight_decay": 0.03,
    "lr_backbone": 1e-5,
    "lr_film": 5e-5,
    "lr_head": 1e-3,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "lora_targets": ["query", "key", "value"],
    "unfreeze_last_n_blocks": 2,
    "label_smoothing": 0.1,
    "mixup_alpha": 0.3,
    "ema_decay": 0.996,
    "kd_lambda": 3.0,
    "kd_temperature": 2.0,
    "old_f1_drop_tolerance": 0.10,
    "patience": 15,
    "data_dir": str(DATA_ROOT),
    "save_dir": str(SAVE_ROOT),
    "v4b_checkpoint": str(DATA_ROOT / "marslandform_v4b_deploy.pt"),
    "files": {
        "tiles_dir": "tiles",
        "mola_features": "mola_features_by_tile.npy",
        "labels": "tile_labels_v5.json",
        "splits": "tile_splits_v5.json",
        "exemplar_buffer": "exemplar_buffer_v5.json",
    },
}

DATA_DIR = Path(CFG["data_dir"])
SAVE_DIR = Path(CFG["save_dir"])
TILES_DIR = DATA_DIR / CFG["files"]["tiles_dir"]
SAVE_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    TILES_DIR,
    DATA_DIR / CFG["files"]["mola_features"],
    DATA_DIR / CFG["files"]["labels"],
    DATA_DIR / CFG["files"]["splits"],
    DATA_DIR / CFG["files"]["exemplar_buffer"],
    Path(CFG["v4b_checkpoint"]),
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("Data directory:", DATA_DIR)
print("Save directory:", SAVE_DIR)
print("Class mapping:", CFG["class_to_idx"])


In [ ]:
class FiLMLayer(nn.Module):
    def __init__(self, mola_dim: int, visual_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.mola_encoder = nn.Sequential(
            nn.BatchNorm1d(mola_dim),
            nn.Linear(mola_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
        )
        self.gamma_proj = nn.Linear(hidden_dim, visual_dim)  # identity init
        self.beta_proj = nn.Linear(hidden_dim, visual_dim)

        nn.init.ones_(self.gamma_proj.bias)
        nn.init.zeros_(self.gamma_proj.weight)
        nn.init.zeros_(self.beta_proj.bias)
        nn.init.zeros_(self.beta_proj.weight)

    def forward(self, visual_features: torch.Tensor, mola_features: torch.Tensor) -> torch.Tensor:
        h = self.mola_encoder(mola_features)
        gamma = self.gamma_proj(h)
        beta = self.beta_proj(h)
        return gamma * visual_features + beta


class FiLMClassifier(nn.Module):
    def __init__(self, visual_dim=768, mola_dim=25, film_hidden=64, head_hidden=128, num_classes=4, dropout=0.4):
        super().__init__()
        self.film = FiLMLayer(mola_dim, visual_dim, film_hidden)
        self.classifier = nn.Sequential(
            nn.Linear(visual_dim, head_hidden),
            nn.BatchNorm1d(head_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, num_classes),
        )

    def forward(self, embeddings: torch.Tensor, mola: torch.Tensor) -> torch.Tensor:
        modulated = self.film(embeddings, mola)
        return self.classifier(modulated)


In [ ]:
from transformers import Dinov2Model
from peft import LoraConfig, get_peft_model


class MarsLandformNetV5(nn.Module):
    def __init__(self, cfg: dict, num_classes: int):
        super().__init__()
        self.cfg = cfg

        self.backbone = Dinov2Model.from_pretrained(cfg["model_name"])
        lora_config = LoraConfig(
            r=cfg["lora_r"],
            lora_alpha=cfg["lora_alpha"],
            lora_dropout=cfg["lora_dropout"],
            target_modules=cfg["lora_targets"],
            bias="none",
        )
        self.backbone = get_peft_model(self.backbone, lora_config)

        for p in self.backbone.parameters():
            p.requires_grad = False
        for name, p in self.backbone.named_parameters():
            if "lora_" in name:
                p.requires_grad = True

        n_unfreeze = cfg["unfreeze_last_n_blocks"]
        if n_unfreeze > 0:
            layers = self.backbone.base_model.model.encoder.layer
            for i in range(len(layers) - n_unfreeze, len(layers)):
                for p in layers[i].parameters():
                    p.requires_grad = True
            print(f"Unfroze last {n_unfreeze} transformer blocks.")

        self.head = FiLMClassifier(
            visual_dim=cfg["hidden_dim"],
            mola_dim=cfg["mola_dim"],
            film_hidden=cfg["film_hidden"],
            head_hidden=cfg["head_hidden"],
            num_classes=num_classes,
            dropout=cfg["dropout"],
        )

    def forward(self, pixel_values: torch.Tensor, mola_features: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.head(cls_token, mola_features)


def build_optimizer(model: nn.Module, cfg: dict):
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]
    film_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("head.film.")]
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("head.classifier.")]

    print(f"Trainable params - backbone: {sum(p.numel() for p in backbone_params):,}")
    print(f"Trainable params - film: {sum(p.numel() for p in film_params):,}")
    print(f"Trainable params - classifier: {sum(p.numel() for p in head_params):,}")

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": cfg["lr_backbone"]},
            {"params": film_params, "lr": cfg["lr_film"]},
            {"params": head_params, "lr": cfg["lr_head"]},
        ],
        weight_decay=cfg["weight_decay"],
    )
    return optimizer


In [ ]:
def extract_state_dict(ckpt_obj: dict) -> dict:
    for key in ["model_state_dict", "state_dict", "student_state_dict"]:
        if isinstance(ckpt_obj, dict) and key in ckpt_obj and isinstance(ckpt_obj[key], dict):
            return ckpt_obj[key]
    if isinstance(ckpt_obj, dict):
        return ckpt_obj
    raise ValueError("Unsupported checkpoint format.")


def split_v4b_state(state_dict: dict):
    backbone_state = {}
    head_state = {}
    for k, v in state_dict.items():
        if k.startswith("backbone."):
            backbone_state[k[len("backbone."):]] = v
        elif k.startswith("head."):
            head_state[k[len("head."):]] = v
        elif k.startswith("film.") or k.startswith("classifier."):
            head_state[k] = v
    return backbone_state, head_state


v4b_ckpt = torch.load(CFG["v4b_checkpoint"], map_location="cpu", weights_only=False)
v4b_state = extract_state_dict(v4b_ckpt)
backbone_state, head_state = split_v4b_state(v4b_state)

for key in ["classifier.4.weight", "classifier.4.bias"]:
    if key not in head_state:
        raise KeyError(f"Missing key in V4b checkpoint: {key}")

teacher = MarsLandformNetV5(CFG, num_classes=4).to(device)
student = MarsLandformNetV5(CFG, num_classes=5).to(device)

teacher_backbone_result = teacher.backbone.load_state_dict(backbone_state, strict=False)
student_backbone_result = student.backbone.load_state_dict(backbone_state, strict=False)
teacher.head.load_state_dict(head_state, strict=True)

head_state_without_last = {k: v for k, v in head_state.items() if k not in {"classifier.4.weight", "classifier.4.bias"}}
student.head.load_state_dict(head_state_without_last, strict=False)

with torch.no_grad():
    old_w = head_state["classifier.4.weight"]
    old_b = head_state["classifier.4.bias"]
    student_last = student.head.classifier[4]

    student_last.weight[:4].copy_(old_w)
    student_last.bias[:4].copy_(old_b)

    nn.init.normal_(student_last.weight[4:5], mean=0.0, std=0.01)
    student_last.bias[4:5].zero_()

max_copy_err = (student.head.classifier[4].weight[:4].cpu() - head_state["classifier.4.weight"]).abs().max().item()
print(f"Copied old logits rows with max error: {max_copy_err:.6e}")

for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()

print("Teacher backbone load - missing:", len(teacher_backbone_result.missing_keys), "unexpected:", len(teacher_backbone_result.unexpected_keys))
print("Student backbone load - missing:", len(student_backbone_result.missing_keys), "unexpected:", len(student_backbone_result.unexpected_keys))
print("Student classifier output dim:", student.head.classifier[4].out_features)
print("Class mapping:", CFG["class_to_idx"])


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def normalize_image(img_tensor: torch.Tensor) -> torch.Tensor:
    return (img_tensor - IMAGENET_MEAN) / IMAGENET_STD


class MarsTileDataset(Dataset):
    def __init__(self, tile_labels, indices, mola_features_by_tile, tiles_dir: Path, class_to_idx: dict, tile_size: int = 224, train: bool = False):
        self.tile_labels = tile_labels
        self.indices = indices
        self.mola_features_by_tile = mola_features_by_tile
        self.tiles_dir = Path(tiles_dir)
        self.class_to_idx = class_to_idx
        self.tile_size = tile_size
        self.train = train
        self.samples = []
        self._build_samples()

    def _resolve_mola(self, image_id, tile_row, tile_col):
        image_keys = [image_id, str(image_id)]
        mola_for_image = None
        for k in image_keys:
            if k in self.mola_features_by_tile:
                mola_for_image = self.mola_features_by_tile[k]
                break
        if mola_for_image is None:
            return None

        candidates = [
            f"{tile_row}_{tile_col}",
            f"{tile_row:03d}_{tile_col:03d}",
            f"{tile_row:04d}_{tile_col:04d}",
        ]
        tuple_key = (tile_row, tile_col)
        if tuple_key in mola_for_image:
            return np.asarray(mola_for_image[tuple_key], dtype=np.float32)

        for key in candidates:
            if key in mola_for_image:
                return np.asarray(mola_for_image[key], dtype=np.float32)
        return None

    def _build_samples(self):
        kept = 0
        skipped = 0
        for idx in self.indices:
            sample = self.tile_labels[idx]
            label_name = str(sample["label"]).upper()
            if label_name not in self.class_to_idx:
                skipped += 1
                continue

            image_id = sample["image_id"]
            tile_row = int(sample["tile_row"])
            tile_col = int(sample["tile_col"])
            img_path = self.tiles_dir / str(image_id) / f"tile_{tile_row:03d}_{tile_col:03d}.jpg"
            if not img_path.exists():
                skipped += 1
                continue

            mola = self._resolve_mola(image_id, tile_row, tile_col)
            if mola is None or mola.shape[0] != 25:
                skipped += 1
                continue

            self.samples.append({
                "img_path": img_path,
                "mola": mola,
                "target": self.class_to_idx[label_name],
                "index": idx,
            })
            kept += 1

        print(f"{'Train' if self.train else 'Eval'} dataset: kept {kept:,}, skipped {skipped:,}")
        if kept > 0:
            counts = Counter(s["target"] for s in self.samples)
            for class_name, class_idx in self.class_to_idx.items():
                n = counts.get(class_idx, 0)
                pct = 100.0 * n / kept
                print(f"  {class_name:>5}: {n:6d} ({pct:5.1f}%)")

    def __len__(self):
        return len(self.samples)

    def _augment(self, img_np):
        if random.random() < 0.5:
            img_np = np.fliplr(img_np).copy()
        if random.random() < 0.5:
            img_np = np.flipud(img_np).copy()
        k = random.randint(0, 3)
        if k > 0:
            img_np = np.rot90(img_np, k=k, axes=(0, 1)).copy()
        return img_np

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["img_path"]).convert("RGB").resize((self.tile_size, self.tile_size), Image.BICUBIC)
        img_np = np.asarray(img, dtype=np.float32) / 255.0

        if self.train:
            img_np = self._augment(img_np)

        img_t = torch.from_numpy(img_np.transpose(2, 0, 1)).float()
        img_t = normalize_image(img_t)
        mola_t = torch.tensor(s["mola"], dtype=torch.float32)
        target = torch.tensor(s["target"], dtype=torch.long)
        return img_t, mola_t, target


with open(DATA_DIR / CFG["files"]["labels"], "r") as f:
    tile_labels = json.load(f)
with open(DATA_DIR / CFG["files"]["splits"], "r") as f:
    tile_splits = json.load(f)
with open(DATA_DIR / CFG["files"]["exemplar_buffer"], "r") as f:
    exemplar_buffer = json.load(f)

mola_features_by_tile = np.load(DATA_DIR / CFG["files"]["mola_features"], allow_pickle=True).item()


def make_incremental_train_indices(tile_labels, split_indices, exemplar_buffer):
    split_set = set(split_indices)
    sct_indices = [i for i in split_indices if str(tile_labels[i]["label"]).upper() == "SCT"]

    old_classes = ["LDA", "LVF", "CCF", "OTHER"]
    exemplar_indices = []
    for cls_name in old_classes:
        exemplar_indices.extend(int(i) for i in exemplar_buffer.get(cls_name, []))

    exemplar_indices = [i for i in exemplar_indices if i in split_set]
    if len(exemplar_indices) == 0:
        exemplar_indices = [i for i in split_indices if str(tile_labels[i]["label"]).upper() in old_classes]
        print("No exemplar indices found in train split; using all old-class train indices.")

    selected = sorted(set(exemplar_indices + sct_indices))
    if len(selected) == 0:
        raise RuntimeError("No train samples selected for incremental training.")
    return selected


train_indices = make_incremental_train_indices(tile_labels, tile_splits["train"], exemplar_buffer)
val_indices = tile_splits["val"]
test_indices = tile_splits["test"]

train_ds = MarsTileDataset(tile_labels, train_indices, mola_features_by_tile, TILES_DIR, CFG["class_to_idx"], tile_size=CFG["tile_size"], train=True)
val_ds = MarsTileDataset(tile_labels, val_indices, mola_features_by_tile, TILES_DIR, CFG["class_to_idx"], tile_size=CFG["tile_size"], train=False)
test_ds = MarsTileDataset(tile_labels, test_indices, mola_features_by_tile, TILES_DIR, CFG["class_to_idx"], tile_size=CFG["tile_size"], train=False)

train_counts = Counter(s["target"] for s in train_ds.samples)
target_probs = {0: 0.125, 1: 0.125, 2: 0.125, 3: 0.125, 4: 0.5}
sample_weights = [target_probs[s["target"]] / max(train_counts[s["target"]], 1) for s in train_ds.samples]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    sampler=sampler,
    num_workers=CFG["num_workers"],
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)

print(f"\nBatches/epoch - train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")


In [ ]:
ce_train = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"], reduction="none")
ce_eval = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])


def kd_loss(student_old_logits: torch.Tensor, teacher_old_logits: torch.Tensor, temperature: float = 2.0) -> torch.Tensor:
    student_log_probs = F.log_softmax(student_old_logits / temperature, dim=1)
    teacher_probs = F.softmax(teacher_old_logits / temperature, dim=1)
    return F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)


def groupwise_mixup(pixel_values, mola_features, targets, alpha=0.3, new_class_idx=4):
    if alpha <= 0:
        lambdas = torch.ones(targets.size(0), device=targets.device)
        return pixel_values, mola_features, targets, targets, lambdas

    mixed_pixels = pixel_values.clone()
    mixed_mola = mola_features.clone()
    targets_a = targets
    targets_b = targets.clone()
    lambdas = torch.ones(targets.size(0), device=targets.device)

    old_idx = torch.where(targets != new_class_idx)[0]
    new_idx = torch.where(targets == new_class_idx)[0]

    for idx in [old_idx, new_idx]:
        if idx.numel() < 2:
            continue
        lam = float(np.random.beta(alpha, alpha))
        perm = idx[torch.randperm(idx.numel(), device=targets.device)]
        mixed_pixels[idx] = lam * pixel_values[idx] + (1.0 - lam) * pixel_values[perm]
        mixed_mola[idx] = lam * mola_features[idx] + (1.0 - lam) * mola_features[perm]
        targets_b[idx] = targets[perm]
        lambdas[idx] = lam

    return mixed_pixels, mixed_mola, targets_a, targets_b, lambdas


def lwf_loss(student_logits, teacher_logits, targets_a, targets_b, lambdas, kd_lambda=3.0, temperature=2.0):
    ce_a = ce_train(student_logits, targets_a)
    ce_b = ce_train(student_logits, targets_b)
    ce = (lambdas * ce_a + (1.0 - lambdas) * ce_b).mean()

    kd = kd_loss(
        student_old_logits=student_logits[:, :4],
        teacher_old_logits=teacher_logits[:, :4],
        temperature=temperature,
    )

    total = kd_lambda * kd + ce
    return total, kd.detach(), ce.detach()


In [ ]:
class EMA:
    def __init__(self, model, decay=0.996):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model):
        for name, param in model.named_parameters():
            if name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}


@torch.no_grad()
def evaluate_model(model, loader, criterion, num_classes=5, old_landform_indices=(0, 1, 2), return_arrays=False):
    model.eval()
    total_loss = 0.0
    y_true, y_pred, y_prob = [], [], []

    for pixel_values, mola_features, targets in loader:
        pixel_values = pixel_values.to(device, non_blocking=True)
        mola_features = mola_features.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(pixel_values, mola_features)
        loss = criterion(logits, targets)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        total_loss += loss.item() * targets.size(0)
        y_true.extend(targets.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())
        y_prob.extend(probs.cpu().tolist())

    avg_loss = total_loss / max(len(loader.dataset), 1)
    labels = list(range(num_classes))
    per_class_f1 = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    old_mask = np.isin(np.asarray(y_true), np.asarray(old_landform_indices))
    if old_mask.any():
        old_true = np.asarray(y_true)[old_mask]
        old_pred = np.asarray(y_pred)[old_mask]
        old_landform_f1 = f1_score(
            old_true,
            old_pred,
            labels=list(old_landform_indices),
            average="macro",
            zero_division=0,
        )
    else:
        old_landform_f1 = 0.0

    metrics = {
        "loss": avg_loss,
        "macro_f1": float(macro_f1),
        "per_class_f1": per_class_f1.tolist(),
        "old_landform_f1": float(old_landform_f1),
        "sct_f1": float(per_class_f1[CFG["new_class_idx"]]),
    }
    if return_arrays:
        metrics["y_true"] = y_true
        metrics["y_pred"] = y_pred
        metrics["y_prob"] = y_prob
    return metrics


@torch.no_grad()
def evaluate_teacher_old_baseline(teacher_model, loader, old_landform_indices=(0, 1, 2)):
    teacher_model.eval()
    y_true, y_pred = [], []
    for pixel_values, mola_features, targets in loader:
        pixel_values = pixel_values.to(device, non_blocking=True)
        mola_features = mola_features.to(device, non_blocking=True)
        logits = teacher_model(pixel_values, mola_features)
        preds = logits.argmax(dim=1).cpu().numpy()
        true = targets.numpy()
        mask = np.isin(true, np.asarray(old_landform_indices))
        y_true.extend(true[mask].tolist())
        y_pred.extend(preds[mask].tolist())
    if len(y_true) == 0:
        return 0.0
    return float(f1_score(y_true, y_pred, labels=list(old_landform_indices), average="macro", zero_division=0))


optimizer = build_optimizer(student, CFG)
total_steps = CFG["num_epochs"] * len(train_loader)
warmup_steps = CFG["warmup_epochs"] * len(train_loader)


def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler(enabled=torch.cuda.is_available())
ema = EMA(student, decay=CFG["ema_decay"])

teacher_old_baseline = evaluate_teacher_old_baseline(teacher, val_loader, tuple(CFG["old_landform_indices"]))
print(f"Teacher baseline old-class macro-F1 (LDA/LVF/CCF): {teacher_old_baseline:.4f}")

history = {
    "train_loss": [],
    "train_kd": [],
    "train_ce": [],
    "val_loss": [],
    "val_macro_f1": [],
    "val_old_landform_f1": [],
    "val_sct_f1": [],
    "lr": [],
}

best_val_macro_f1 = -1.0
best_epoch = -1
best_state = None
patience_counter = 0

for epoch in range(1, CFG["num_epochs"] + 1):
    student.train()
    running_loss = 0.0
    running_kd = 0.0
    running_ce = 0.0
    n_samples = 0

    for pixel_values, mola_features, targets in train_loader:
        pixel_values = pixel_values.to(device, non_blocking=True)
        mola_features = mola_features.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        mix_pixels, mix_mola, y_a, y_b, lambdas = groupwise_mixup(
            pixel_values,
            mola_features,
            targets,
            alpha=CFG["mixup_alpha"],
            new_class_idx=CFG["new_class_idx"],
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            teacher_logits = teacher(mix_pixels, mix_mola)

        with autocast(enabled=torch.cuda.is_available()):
            student_logits = student(mix_pixels, mix_mola)
            loss, kd_component, ce_component = lwf_loss(
                student_logits,
                teacher_logits,
                y_a,
                y_b,
                lambdas,
                kd_lambda=CFG["kd_lambda"],
                temperature=CFG["kd_temperature"],
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(student)

        bs = targets.size(0)
        running_loss += loss.item() * bs
        running_kd += kd_component.item() * bs
        running_ce += ce_component.item() * bs
        n_samples += bs

    train_loss = running_loss / max(1, n_samples)
    train_kd = running_kd / max(1, n_samples)
    train_ce = running_ce / max(1, n_samples)

    ema.apply_shadow(student)
    val_metrics = evaluate_model(
        student,
        val_loader,
        criterion=ce_eval,
        num_classes=CFG["num_classes"],
        old_landform_indices=tuple(CFG["old_landform_indices"]),
        return_arrays=False,
    )
    if val_metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        best_state = copy.deepcopy(student.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
    ema.restore(student)

    history["train_loss"].append(train_loss)
    history["train_kd"].append(train_kd)
    history["train_ce"].append(train_ce)
    history["val_loss"].append(val_metrics["loss"])
    history["val_macro_f1"].append(val_metrics["macro_f1"])
    history["val_old_landform_f1"].append(val_metrics["old_landform_f1"])
    history["val_sct_f1"].append(val_metrics["sct_f1"])
    history["lr"].append(optimizer.param_groups[0]["lr"])

    print(
        f"Epoch {epoch:02d}/{CFG['num_epochs']} | "
        f"train={train_loss:.4f} (kd={train_kd:.4f}, ce={train_ce:.4f}) | "
        f"val_f1={val_metrics['macro_f1']:.4f} | "
        f"old_f1={val_metrics['old_landform_f1']:.4f} | "
        f"sct_f1={val_metrics['sct_f1']:.4f}"
    )

    # old_f1 early stopping disabled — KD loss handles preservation

    if patience_counter >= CFG["patience"]:
        print(f"Early stopping: no val macro-F1 improvement for {CFG['patience']} epochs.")
        break

if best_state is None:
    ema.apply_shadow(student)
    best_state = copy.deepcopy(student.state_dict())
    ema.restore(student)
    best_epoch = epoch
    best_val_macro_f1 = history["val_macro_f1"][-1]

BEST_CKPT_PATH = SAVE_DIR / "marslandform_v5_best.pt"
torch.save(
    {
        "model_state_dict": best_state,
        "cfg": CFG,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "teacher_old_baseline": teacher_old_baseline,
    },
    BEST_CKPT_PATH,
)

with open(SAVE_DIR / "history_v5_lwf.json", "w") as f:
    json.dump(history, f)

print(f"\nBest epoch: {best_epoch}, val macro-F1: {best_val_macro_f1:.4f}")
print(f"Saved best checkpoint: {BEST_CKPT_PATH}")


In [ ]:
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt["model_state_dict"], strict=True)
student.eval()

TEST_METRICS = evaluate_model(
    student,
    test_loader,
    criterion=ce_eval,
    num_classes=CFG["num_classes"],
    old_landform_indices=tuple(CFG["old_landform_indices"]),
    return_arrays=True,
)

print("=" * 72)
print("MarsLandformNet V5 Test Metrics")
print("=" * 72)
for class_name, class_f1 in zip(CFG["class_names"], TEST_METRICS["per_class_f1"]):
    print(f"{class_name:>5} F1: {class_f1:.4f}")
print(f"Macro F1 (5-class): {TEST_METRICS['macro_f1']:.4f}")
print(f"Old-class F1 (LDA+LVF+CCF avg): {TEST_METRICS['old_landform_f1']:.4f}")
print(f"New-class F1 (SCT): {TEST_METRICS['sct_f1']:.4f}")
print("=" * 72)

print(
    classification_report(
        TEST_METRICS["y_true"],
        TEST_METRICS["y_pred"],
        labels=list(range(CFG["num_classes"])),
        target_names=CFG["class_names"],
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(
    TEST_METRICS["y_true"],
    TEST_METRICS["y_pred"],
    labels=list(range(CFG["num_classes"])),
)
cm_pct = cm.astype(np.float64) / np.clip(cm.sum(axis=1, keepdims=True), a_min=1, a_max=None) * 100.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CFG["class_names"],
    yticklabels=CFG["class_names"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix (Counts)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(
    cm_pct,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=CFG["class_names"],
    yticklabels=CFG["class_names"],
    ax=axes[1],
)
axes[1].set_title("Confusion Matrix (%)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.savefig(SAVE_DIR / "confusion_matrix_v5_lwf.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
student.eval()

if hasattr(student.backbone, "merge_and_unload"):
    merged_backbone = student.backbone.merge_and_unload()
    print("Merged LoRA adapters into backbone weights.")
else:
    merged_backbone = student.backbone
    print("Backbone has no merge_and_unload; saving current backbone weights.")

deploy_model_state = {}
for k, v in merged_backbone.state_dict().items():
    deploy_model_state[f"backbone.base_model.model.{k}"] = v.detach().cpu()

for k, v in student.head.state_dict().items():
    deploy_model_state[k] = v.detach().cpu()

deploy_state = {
    "model_state_dict": deploy_model_state,
    "cfg": CFG,
    "class_names": CFG["class_names"],
    "class_to_idx": CFG["class_to_idx"],
    "old_class_indices": CFG["old_class_indices"],
    "new_class_idx": CFG["new_class_idx"],
    "best_epoch": int(best_ckpt["best_epoch"]),
    "best_val_macro_f1": float(best_ckpt["best_val_macro_f1"]),
    "test_macro_f1": float(TEST_METRICS["macro_f1"]),
    "test_old_landform_f1": float(TEST_METRICS["old_landform_f1"]),
    "test_sct_f1": float(TEST_METRICS["sct_f1"]),
    "version": "v5-lwf-sct-expansion",
    "lora_merged": True,
    "teacher_checkpoint": CFG["v4b_checkpoint"],
}

DEPLOY_PATH = SAVE_DIR / "marslandform_v5_deploy.pt"
torch.save(deploy_state, DEPLOY_PATH)

print(f"Deploy checkpoint exported: {DEPLOY_PATH}")
print(f"Size: {DEPLOY_PATH.stat().st_size / 1e6:.1f} MB")
print("Checkpoint format: backbone.base_model.model.* + film.* + classifier.*")
